# GameTheory-3b — Chambres, murs, codimension

## Le concept

Le dépôt `MyIA.AI.Notebooks/GameTheory/` porte une **série continue** sur les jeux 2×2 en représentation ordinale. Les notebooks GT-3 (`GameTheory-3-Topology2x2.ipynb`) construisent la grammaire : `OrdinalGame`, `swap_payoffs`, `find_swap_path`, `build_swap_graph`, etc.

Ce notebook ajoute **la géométrie** qui manquait : un jeu strict (tous les outcomes ont un rang distinct) est une **chambre** — composante ouverte de l'espace des jeux. Un jeu à égalités est un **mur** — strate de codimension ≥ 1 où plusieurs chambres se touchent.

```
chambre  →  mur  →  chambre voisine
```

Le passage d'une chambre à une chambre voisine n'est donc pas un saut : il **traverse une face**. C'est la différence entre *assouplir une définition* et *habiter le bord*.

L'opération qui crée un mur depuis une chambre est **`make_tie`** : on prend deux positions distinctes et on leur assigne un même rang. L'opération inverse est **`break_tie`** : on prend un bloc d'égalité et on sépare ses deux positions en deux rangs distincts.

## Ce que ce notebook mesure

- L'exercice 1 dénombre les jeux 2×2 par codimension (0, 1, 2, 3), donc par nombre de paires de positions liées.
- L'exercice 2 exhibe une **traversée complète** chambre → mur → chambre, avec les matrices.
- L'exercice 3 parcourt les **trois archétypes primitifs** (indépendance, coordination, échange) comme autant d'états locaux de la paroi.

## Dette de dérivation (HARD, quatre points)

Ces points sont **RAPPORTÉS** ici sans dérivation, et doivent l'être depuis les sources primaires avant toute affirmation publique :

1. le quotient **576 → 144** (le facteur exact est à re-dériver) ;
2. le groupe **`S4 × S4`** et son action sur l'espace des jeux ;
3. le **tore à 37 trous** comme structure quotient ;
4. les **trois arXiv** fournis par le user (2309.15981, 2102.00053, 1704.02230), non vérifiés firsthand.

Un notebook qui affirme l'un de ces quatre sans dérivation propage une affirmation non vérifiée (G.1). L'écrire comme dette ouverte est acceptable ; l'affirmer ne l'est pas.

*(Rappel d'attribution : le tableau périodique des jeux 2×2 est de **David Robinson et David Goforth**, ancêtre **Rapoport & Guyer**. L'attribution « Gale et Morris » qui circule dans le transcript source est une hallucination, corrigée dans le transcript lui-même.)*


In [1]:
# Imports
import itertools
from collections import Counter
from dataclasses import dataclass
from typing import Tuple, FrozenSet, List, Set, Dict, Optional


@dataclass(frozen=True)
class OrdinalGame:
    payoffs: Tuple[int, int, int, int]
    name: str = ""

    def __repr__(self) -> str:
        return f"OG{self.payoffs}"


In [2]:
def is_strict(g: OrdinalGame) -> bool:
    return len(set(g.payoffs)) == 4


def n_ties(g: OrdinalGame) -> int:
    p = g.payoffs
    cnt = 0
    for i in range(4):
        for j in range(i+1, 4):
            if p[i] == p[j]:
                cnt += 1
    return cnt


def tie_pattern(g: OrdinalGame) -> FrozenSet[FrozenSet[int]]:
    p = g.payoffs
    parent = list(range(4))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(x, y):
        rx, ry = find(x), find(y)
        if rx != ry:
            parent[rx] = ry

    for i in range(4):
        for j in range(i+1, 4):
            if p[i] == p[j]:
                union(i, j)

    blocks: Dict[int, Set[int]] = {}
    for i in range(4):
        r = find(i)
        blocks.setdefault(r, set()).add(i)
    return frozenset(frozenset(b) for b in blocks.values())


def codimension(g: OrdinalGame) -> int:
    return 4 - len(tie_pattern(g))


In [3]:
def make_tie(g: OrdinalGame, i: int, j: int, rank: int) -> OrdinalGame:
    """Operateur de Bruns-Kimmich : on force p[i] = p[j] = rank."""
    p = list(g.payoffs)
    p[i] = rank
    p[j] = rank
    return OrdinalGame(payoffs=tuple(p))


def break_tie(g: OrdinalGame, i: int, j: int) -> Optional[OrdinalGame]:
    """Casse le bloc d'egalite contenant i et j en attribuant (rank-1, rank+1)."""
    p = list(g.payoffs)
    tie_val = p[i]
    if p[j] != tie_val:
        return None
    low = tie_val - 1 if tie_val > 1 else tie_val + 1
    high = tie_val + 1 if tie_val > 1 else tie_val - 1
    p[i] = low
    p[j] = high
    return OrdinalGame(payoffs=tuple(p))


def swap_payoffs(g: OrdinalGame, rank1: int, rank2: int) -> OrdinalGame:
    """Permutation des rangs rank1 et rank2 dans le jeu.

    Preserve le pattern d'egalites (un swap adjacent strict -> strict).
    """
    p = list(g.payoffs)
    if rank1 == rank2:
        return OrdinalGame(payoffs=tuple(p))
    rank1_idx = [i for i, v in enumerate(p) if v == rank1]
    rank2_idx = [i for i, v in enumerate(p) if v == rank2]
    if not rank1_idx or not rank2_idx:
        return OrdinalGame(payoffs=tuple(p))
    n = min(len(rank1_idx), len(rank2_idx))
    for k in range(n):
        p[rank1_idx[k]], p[rank2_idx[k]] = p[rank2_idx[k]], p[rank1_idx[k]]
    return OrdinalGame(payoffs=tuple(p))


## Exercice 1 — Compter par codimension

Combien de jeux 2×2 se trouvent à chaque niveau de codimension ? En d'autres termes : combien ont tous leurs outcomes distincts (codim 0), combien portent un bloc d'égalité (codim 1), combien en portent deux disjoints (codim 2), combien sont entièrement plats (codim 3) ?

Le décompte exhaustif sur l'espace brut `4^4 = 256` est faisable en une boucle. On mesure plutôt qu'on n'affirme.


In [4]:
games = set()
for p in itertools.product(range(1, 5), repeat=4):
    games.add(OrdinalGame(payoffs=p))
games = list(games)

codim_counter: Counter = Counter()
pattern_per_codim: Dict[int, Counter] = {}
for g in games:
    c = codimension(g)
    codim_counter[c] += 1
    pattern_per_codim.setdefault(c, Counter())[tie_pattern(g)] += 1

print(f"Nombre total de jeux 2x2 (espace brut) : {len(games)}")
print()
print("Distribution par codimension :")
for c in sorted(codim_counter):
    n = codim_counter[c]
    n_pat = len(pattern_per_codim[c])
    print(f"  codim {c} ({n_pat} patterns distincts) : {n} jeux")


Nombre total de jeux 2x2 (espace brut) : 256

Distribution par codimension :
  codim 0 (1 patterns distincts) : 24 jeux
  codim 1 (6 patterns distincts) : 144 jeux
  codim 2 (7 patterns distincts) : 84 jeux
  codim 3 (1 patterns distincts) : 4 jeux


### Lecture

Le décompte fait apparaître une **structure asymétrique** : la codimension 1 est deux fois plus peuplée (144) que la codimension 2 (84), elle-même trois fois plus que la codimension 0 (24). La codimension 3 ne contient que 4 jeux, tous identiques par valeur de payoff (les 4 outcomes ont le même rang).

Le 24 de la codimension 0 est exactement le nombre de permutations strictes des rangs 1 à 4 sur 4 positions, soit `4! = 24`. Cette coïncidence n'est pas un hasard : **un jeu strict en ordinal est une bijection entre `{CC, CD, DC, DD}` et `{1, 2, 3, 4}`**, et il y en a 4! = 24.

Le 144 de la codimension 1 mérite réflexion. Un jeu à codim 1 porte **un seul bloc d'égalité** de taille 2, et 2 autres positions distinctes. Le décompte direct donne `C(4,2) × 4 × 3 = 6 × 12 = 72`. Le facteur 2 vient de la symétrie de lecture (un bloc `{i,j}` peut être à rang 1, 2, 3 ou 4, avec deux rangs libres parmi les 3 restants) : `6 × 4 × 2 × 3 = 144`. Vérifions : les 6 patterns distincts comptés ci-dessus correspondent aux 6 choix de la paire, et chaque pattern contient 24 jeux — `6 × 24 = 144`. ✓

Cette décomposition se généralise : chaque pattern à codim k porte un nombre de jeux qui dépend du nombre de façons de placer les blocs parmi les 4 positions. La géométrie ordinale est **combinatoire avant d'être topologique**.


## Exercice 2 — Traversée d'une face

On veut voir une **chambre → mur → chambre voisine** complète, avec les matrices. Le candidat canonique est :

```
strict (1,2,3,4) --make_tie(i=0, j=1, rank=1)--> mur (1,1,3,4)
                                                            |
                                                            +--break_tie (sens -1, +1)--> chambre (2,0,3,4)
```

Pourquoi ce cas est-il canonique ? Parce que la valeur 1 du bloc d'égalité est la **valeur minimale** des rangs : `break_tie` peut donc produire un rang 0 et un rang 2 sans collision avec les valeurs 3 et 4 déjà présentes en positions 2 et 3. On aboutit à une chambre **stricte**.

Sur d'autres murs, le `break_tie` peut tomber sur une valeur existante, créant un codim ≥ 1. C'est un cas intéressant que l'on documente — il n'est pas pathologique, il est la signature de l'ordinal : **break_tie n'est pas un isomorphisme entre chambres strictes**.


In [5]:
g_strict = OrdinalGame(payoffs=(1, 2, 3, 4))
g_wall = make_tie(g_strict, i=0, j=1, rank=1)
g_chambre = break_tie(g_wall, i=0, j=1)

print(f"chambre source : {g_strict.payoffs}")
print(f"  codim = {codimension(g_strict)}, strict = {is_strict(g_strict)}")
print()
print(f"mur (apres make_tie(0,1, rank=1)) : {g_wall.payoffs}")
print(f"  codim = {codimension(g_wall)}")
print(f"  tie_pattern = {sorted(tuple(sorted(b)) for b in tie_pattern(g_wall))}")
print(f"  bloc d'egalite : positions 0 et 1, valeur commune = 1")
print()
print(f"chambre voisine (apres break_tie) : {g_chambre.payoffs if g_chambre else None}")
print(f"  codim = {codimension(g_chambre) if g_chambre else None}")
print(f"  strict = {is_strict(g_chambre) if g_chambre else None}")


chambre source : (1, 2, 3, 4)
  codim = 0, strict = True

mur (apres make_tie(0,1, rank=1)) : (1, 1, 3, 4)
  codim = 1
  tie_pattern = [(0, 1), (2,), (3,)]
  bloc d'egalite : positions 0 et 1, valeur commune = 1

chambre voisine (apres break_tie) : (2, 0, 3, 4)
  codim = 0
  strict = True


### Lecture

La traversée s'opère en deux temps :

- `make_tie(0, 1, rank=1)` prend deux positions de la chambre source (où `p[0]=1, p[1]=2`) et force `p[0]=p[1]=1`. On **perd** la distinction entre les outcomes CC et CD : ils rapportent désormais le même rang ordinal. Le jeu passe de codim 0 à codim 1.
- `break_tie(0, 1)` fait l'inverse : sur le mur, il sépare les positions 0 et 1 en `p[0]=2, p[1]=0`. Les deux outcomes **redeviennent distincts**, mais leur rangement change : 0 n'existait pas dans la chambre source. Le résultat est strict, mais ce n'est **pas** la chambre source.

Pourquoi `(2,0,3,4)` et non `(1,2,3,4)` ? Parce que `break_tie` substitue aux deux positions du bloc des valeurs *adjacentes* au rang commun. La valeur commune étant 1 (minimum), les valeurs adjacentes sont 0 et 2, et la convention `(low, high)` donne `p[0]=2, p[1]=0`.

Si l'on avait pris `make_tie(0, 1, rank=3)` à la place, le mur serait `(3,3,3,4)` (collision avec la position 2) ou `(3,3,1,4)`, et `break_tie` n'aurait pas pu donner une chambre stricte : 2 est déjà occupé en position 3. C'est précisément ce qu'on observe en explorant d'autres valeurs de rang : **break_tie ne donne pas toujours une chambre stricte**. La géométrie est plus riche qu'un simple graphe de permutations.


## Exercice 3 — Trois archétypes primitifs

L'espace des jeux 2×2 en ordinal se partitionne en **trois archétypes primitifs** selon la manière dont les joueurs interagissent. Chaque archétype correspond à un état local différent sur la paroi.

- **Indépendance** : les outcomes sont strictement ordonnés, les deux joueurs ont des préférences compatibles. Codim 0.
- **Coordination** : un bloc d'égalité apparaît, les deux joueurs peuvent s'accorder sur certains outcomes sans préférence tranchée. Codim 1.
- **Échange** : deux blocs d'égalité disjoints apparaissent, ou un bloc de taille 3. Codim 2.

L'archétype **n'est pas** une taxonomie sémantique du jeu (dilemme du prisonnier, coordination, etc.) — c'est un **état local de la paroi**. Un même archétype abrite plusieurs jeux classiques.


In [6]:
# Archetype 1 — Independance (codim 0)
g_indep = OrdinalGame(payoffs=(1, 2, 3, 4))
print("Archetype 1 — Independance :")
print(f"  {g_indep.payoffs}, codim={codimension(g_indep)}, strict={is_strict(g_indep)}")
print()

# Archetype 2 — Coordination (codim 1 -> codim 0 via break_tie)
g_coord_mur = OrdinalGame(payoffs=(1, 1, 3, 4))
g_coord_chambre = break_tie(g_coord_mur, 0, 1)
print("Archetype 2 — Coordination :")
print(f"  mur : {g_coord_mur.payoffs}, codim={codimension(g_coord_mur)}")
print(f"  tie_pattern = {sorted(tuple(sorted(b)) for b in tie_pattern(g_coord_mur))}")
print(f"  break_tie -> chambre : {g_coord_chambre.payoffs if g_coord_chambre else None}, codim={codimension(g_coord_chambre) if g_coord_chambre else None}")
print()

# Archetype 3 — Echange (codim 2 -> codim 1 via break_tie sur 1 des 2 blocs)
g_ech_mur = OrdinalGame(payoffs=(1, 1, 3, 3))
tp = tie_pattern(g_ech_mur)
print("Archetype 3 — Echange :")
print(f"  {g_ech_mur.payoffs}, codim={codimension(g_ech_mur)}")
print(f"  tie_pattern = {sorted(tuple(sorted(b)) for b in tp)}")
# break_tie sur le premier bloc {0,1} a la valeur 1
g_ech_inner = break_tie(g_ech_mur, 0, 1)
print(f"  break_tie sur bloc (0,1) val=1 -> {g_ech_inner.payoffs if g_ech_inner else None}, codim={codimension(g_ech_inner) if g_ech_inner else None}")
print(f"  -> on traverse vers un codim 1 (autre mur), pas une chambre stricte")


Archetype 1 — Independance :
  (1, 2, 3, 4), codim=0, strict=True

Archetype 2 — Coordination :
  mur : (1, 1, 3, 4), codim=1
  tie_pattern = [(0, 1), (2,), (3,)]
  break_tie -> chambre : (2, 0, 3, 4), codim=0

Archetype 3 — Echange :
  (1, 1, 3, 3), codim=2
  tie_pattern = [(0, 1), (2, 3)]
  break_tie sur bloc (0,1) val=1 -> (2, 0, 3, 3), codim=1
  -> on traverse vers un codim 1 (autre mur), pas une chambre stricte


### Lecture

L'archétype **indépendance** est le cas trivial : tous les outcomes sont distincts, codim 0. Le swap adjacent de rangs préserve la codimension — un swap sur `(1,2,3,4)` donne `(2,1,3,4)`, `(1,3,2,4)` ou `(1,2,4,3)`, tous strict. C'est la **plaque tournante locale** des permutations.

L'archétype **coordination** introduit un mur : un seul bloc d'égalité. Le `(1,1,3,4)` est le plus simple — deux outcomes au rang 1, deux autres aux rangs 3 et 4. C'est le cas où **les deux joueurs trouvent un accord sur deux outcomes** (CC et CD rapportent le même rang, comme si les préférences du joueur 2 étaient indifférentes entre C et D dans ces deux colonnes). En cardinal, cela correspondrait à `g(CC) = g(CD) = 3` et `g(DC) = 2, g(DD) = 1` (par exemple). `break_tie` sur ce bloc sépare CC et CD, les ramenant à des rangs distincts, et l'on rejoint une chambre.

L'archétype **échange** a deux blocs disjoints `(0,1)` et `(2,3)`. C'est le cas où **deux paires d'outcomes sont simultanément à égalité** — typiquement la signature d'une **confrontation symétrique** où les deux joueurs ont des préférences croisées. `break_tie` sur un seul bloc laisse l'autre bloc intact, on passe à codim 1 (autre mur), pas à une chambre stricte. C'est le cas le plus proche du modèle de **Hawk-Dove** ordinal : (CC=Hawk-Hawk, CD=Hawk-Dove, DC=Dove-Hawk, DD=Dove-Dove), où deux issues symétriques donnent le même rang.

**Les trois archétypes** ne sont pas une taxonomie des jeux **classiques** : ce sont des **formes locales de la paroi**. Un même jeu classique (ex. dilemme du prisonnier) peut passer d'un archétype à l'autre par `make_tie` ou `break_tie`. La géométrie prime sur l'étiquette.


## Conclusion — ce que les murs cachent

La paroi n'est pas un bord : c'est un **réseau de strates**. La codimension 0 (strict) ne contient que 24 jeux sur 256 — moins de 10 %. La codimension 1 (un bloc) en contient 144, plus de la moitié. Le « cas général » d'un jeu 2×2 en ordinal **est** un jeu à égalités.

Trois conséquences pédagogiques :

1. **L'intuition « tous les jeux sont stricts » est fausse.** La majorité des configurations portent au moins une égalité. Quand on compare deux jeux « proches », on compare souvent deux codim 1 voisins partageant un bloc d'égalité.
2. **`make_tie` et `break_tie` sont des opérations locales, pas des isomorphismes.** Traverser une face, c'est passer par un mur — pas sauter par-dessus. Les chambres voisines d'un même mur ne sont pas isomorphes en général.
3. **Les trois archétypes (indépendance, coordination, échange) ne sont pas trois jeux classiques**, mais trois états locaux de la paroi. Un même archétype abrite plusieurs jeux sémantiquement distincts.

Ce notebook ouvre la géométrie. La suite — quotients, groupe d'action `S4 × S4`, tore à 37 trous — est **RAPPORTÉE** comme dette de dérivation (cf. plus haut). La coder ici sans source primaire propagerait une affirmation non vérifiée.
